In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz, csr_matrix



In [6]:
Gdrug_mu = pd.read_pickle("./data/Gdrug_mu.pkl")
Gdrug_sigma = pd.read_pickle("./data/Gdrug_sigma.pkl")

drug_index = pd.read_csv("./data/drug_index.csv")           # should align with Gdrug_* rows
gene_index = pd.read_csv("./data/gene_index_lincs.csv")     # should align with Gdrug_* cols

print("Gdrug_mu:", Gdrug_mu.shape)
print("Gdrug_sigma:", Gdrug_sigma.shape)
print("drug_index:", drug_index.shape)
print("gene_index:", gene_index.shape)


Gdrug_mu: (978, 679)
Gdrug_sigma: (978, 679)
drug_index: (679, 2)
gene_index: (978, 2)


In [7]:
# Transpose gene×drug → drug×gene
Gdrug_mu_T = Gdrug_mu.T
Gdrug_sigma_T = Gdrug_sigma.T

print("After transpose:")
print("Gdrug_mu_T:", Gdrug_mu_T.shape)       # (679, 978)
print("Gdrug_sigma_T:", Gdrug_sigma_T.shape)


After transpose:
Gdrug_mu_T: (679, 978)
Gdrug_sigma_T: (679, 978)


In [8]:
assert Gdrug_mu_T.shape[0] == drug_index.shape[0]
assert Gdrug_mu_T.shape[1] == gene_index.shape[0]


In [9]:
Gdrug_mu_T.to_pickle("./data/Gdrug_mu_drugxgene.pkl")
Gdrug_sigma_T.to_pickle("./data/Gdrug_sigma_drugxgene.pkl")

In [10]:
Gdrug_mu = pd.read_pickle("Gdrug_mu_drugxgene.pkl")
Gdrug_sigma = pd.read_pickle("Gdrug_sigma_drugxgene.pkl")

Gadr = load_npz("Gadr.npz")

gene_index = pd.read_csv("gene_index_lincs.csv")

print(Gdrug_mu.shape)   # (n_drug, n_gene)
print(Gadr.shape)       # (n_adr, n_gene)

(679, 978)
(3659, 978)


In [12]:
drug_reachable = (Gdrug_mu.fillna(0).to_numpy() != 0).any(axis=0)
keep_idx = np.where(drug_reachable)[0]

print("Drug-reachable genes:", len(keep_idx))

Drug-reachable genes: 978


In [13]:
Gdrug_mu = Gdrug_mu.iloc[:, keep_idx]
Gdrug_sigma = Gdrug_sigma.iloc[:, keep_idx]
Gadr = Gadr[:, keep_idx]
gene_index = gene_index.iloc[keep_idx].reset_index(drop=True)

In [14]:
adr_degree = np.asarray((Gadr != 0).sum(axis=0)).ravel()

cutoff = np.percentile(adr_degree, 99)   # drop top 1%
keep_idx = np.where(adr_degree <= cutoff)[0]

print("Dropping hub genes:", (adr_degree > cutoff).sum())

Dropping hub genes: 10


In [15]:
Gdrug_mu = Gdrug_mu.iloc[:, keep_idx]
Gdrug_sigma = Gdrug_sigma.iloc[:, keep_idx]
Gadr = Gadr[:, keep_idx]
gene_index = gene_index.iloc[keep_idx].reset_index(drop=True)

In [16]:
drug_signal = (Gdrug_mu.fillna(0).to_numpy() != 0).any(axis=0)
adr_signal = np.asarray((Gadr != 0).sum(axis=0)).ravel() > 0

keep_idx = np.where(drug_signal & adr_signal)[0]

print("Final genes:", len(keep_idx))


Final genes: 574


In [17]:
Gdrug_mu = Gdrug_mu.iloc[:, keep_idx]
Gdrug_sigma = Gdrug_sigma.iloc[:, keep_idx]
Gadr = Gadr[:, keep_idx]
gene_index = gene_index.iloc[keep_idx].reset_index(drop=True)

In [18]:
from scipy.sparse import save_npz

Gdrug_mu.to_pickle("Gdrug_mu_restricted.pkl")
Gdrug_sigma.to_pickle("Gdrug_sigma_restricted.pkl")
save_npz("Gadr_restricted.npz", Gadr)

gene_index.to_csv("gene_index_lincs_restricted.csv", index=False)

In [19]:
# Restricted drug–gene
Gdrug_mu = pd.read_pickle("Gdrug_mu_restricted.pkl")
Gdrug_sigma = pd.read_pickle("Gdrug_sigma_restricted.pkl")

# Restricted ADR–gene
Gadr = load_npz("Gadr_restricted.npz")

# Indices
drug_index = pd.read_csv("drug_index.csv")                  # unchanged
gene_index = pd.read_csv("gene_index_lincs_restricted.csv") # restricted
adr_index  = pd.read_csv("adr_index.csv")                   # unchanged

In [20]:
n_drugs = Gdrug_mu.shape[0]
n_genes = Gdrug_mu.shape[1]
n_adrs  = Gadr.shape[0]

print("FINAL COUNTS")
print("Drugs :", n_drugs)
print("Genes :", n_genes)
print("ADRs  :", n_adrs)


FINAL COUNTS
Drugs : 679
Genes : 574
ADRs  : 3659
